# Ports & Logistics AI — RAG Pipeline

This notebook implements the RAG pipeline for the Ports & Logistics AI project.

## Pipeline

Inventory
→ Extraction
→ Metadata
→ Cleaning
→ Chunking
→ Embeddings
→ Qdrant
→ Retrieval
→ RBAC
→ Evaluation

## Project Architecture

- PostgreSQL/PostGIS → structured operational data
- MongoDB → semi-structured operational data
- Qdrant → vector retrieval / RAG
- LLM → reasoning and answer generation

## Working Principle

Each stage is implemented, inspected, and validated before moving
to the next stage.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\Users\terbo\Desktop\Project\Data\raw")
RAW_DATA = PROJECT_ROOT / "unstructured"

print("Python:", sys.version)
print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA)
print("Raw data exists:", RAW_DATA.exists())

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
Project root: C:\Users\terbo\Desktop\Project\Data\raw
Raw data: C:\Users\terbo\Desktop\Project\Data\raw\unstructured
Raw data exists: True


## 1. RAG Corpus Inventory

Before extracting text or creating embeddings, we inspect the
available corpus and record its physical properties.

We do not start with embeddings because document identity,
file validity, page count, and metadata must be established first.

In [2]:
pdf_files = sorted(RAW_DATA.rglob("*.pdf"))

print(f"PDF files found: {len(pdf_files)}")

for file in pdf_files:
    print(file.name)

PDF files found: 30
FBL_template.pdf
HbR_Annual_report_2025_highlights.pdf
LPI.pdf
LPI_Methodology.pdf
MSC-MEPC_3-Circ_4-Rev_1.pdf
Transit_Dues_Rates.pdf
BIG_STICK_7895450_ROI_Redacted.pdf
CAROL_ANN_7816410_ROI_Redacted.pdf
CONCHRD_7107811_ROI_Redacted.pdf
Critical_Port_Infrastructure_Risks_PWSA_BOIR_Redacted.pdf
CUMBERLAND_II_8126294_ROI_Redacted.pdf
CUYAHOGA_7881082_ROI_Redacted.pdf
GENERAL_MACARTHUR_7861135_ROI_Redacted.pdf
GOOD_GULF_8194633_ROI_Redacted.pdf
H20_BELOW_6966090_ROI_Redacted.pdf
KIM_LY_7867677_ROI_Redacted.pdf
LA6893ER_8027803_ROI_Redacted.pdf
LADY_ELLEN_7965695_ROI_Redacted.pdf
LITTLE_G_7717011_ROI_Redacted.pdf
MERRICK_8031230_ROI_Redacted.pdf
MICHIPICOTEN_7971638_ROI_Redacted.pdf
NC3250BW_7987052_ROI_Redacted.pdf
PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf
QUEEN_OF_THE_WEST_4405289_ROI_Redacted.pdf
SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf
TOO_SHORT_7943023_ROI_Redacted.pdf
UNA_MAS_8231938_ROI_Redacted.pdf
VICAR_602_7989592_ROI_Redacted.pdf
WAYMON_BOYD_7039383_ROI_Re

In [3]:
import pypdf
import pandas as pd

inventory = []

for file in pdf_files:
    reader = pypdf.PdfReader(str(file))

    inventory.append({
        "filename": file.name,
        "extension": file.suffix.lower(),
        "size_mb": round(file.stat().st_size / (1024 * 1024), 2),
        "page_count": len(reader.pages),
        "path": str(file)
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,filename,extension,size_mb,page_count,path
0,FBL_template.pdf,.pdf,0.17,3,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
1,HbR_Annual_report_2025_highlights.pdf,.pdf,18.65,85,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
2,LPI.pdf,.pdf,1.00,30,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
3,LPI_Methodology.pdf,.pdf,0.96,32,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
4,MSC-MEPC_3-Circ_4-Rev_1.pdf,.pdf,0.47,31,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
5,Transit_Dues_Rates.pdf,.pdf,0.60,4,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
6,BIG_STICK_7895450_ROI_Redacted.pdf,.pdf,4.48,18,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
7,CAROL_ANN_7816410_ROI_Redacted.pdf,.pdf,2.40,51,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
8,CONCHRD_7107811_ROI_Redacted.pdf,.pdf,3.41,45,C:\Users\terbo\Desktop\Project\Data\raw\unstru...
9,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,.pdf,0.97,55,C:\Users\terbo\Desktop\Project\Data\raw\unstru...


## 1.1 Document Classification

Each document is classified according to its role in the RAG corpus.

Classification is based on the project's Master Checkpoint and the
actual corpus inventory.

The classification fields are:

- source
- real_or_synthetic
- document_type
- department
- sensitivity_level
- version_date

## 1.2 Classification Rules

The classification is based on the verified corpus and the Master Checkpoint.

For the 24 USCG reports, the documents are classified as real regulatory
marine casualty reports. They provide general maritime-safety knowledge and
are not tied to the project's synthetic operational data.

The Suez Canal tariff PDF has a special ingestion scope:
only its Notes section is relevant to RAG. The numeric tariff tables remain
in PostgreSQL.

Version/date values are only populated when they are explicitly known from
the document or project context. Unknown values remain null rather than
being guessed.

In [4]:
# Document classification metadata

core_classification = {
    "FBL_template.pdf": {
        "source": "FIATA",
        "real_or_synthetic": "real",
        "document_type": "bill_of_lading_template",
        "department": "Operations",
        "sensitivity_level": 1,
        "version_date": None,
        "ingestion_scope": "full_document",
    },

    "HbR_Annual_report_2025_highlights.pdf": {
        "source": "Port of Rotterdam Authority",
        "real_or_synthetic": "real",
        "document_type": "annual_report",
        "department": "Operations",
        "sensitivity_level": 1,
        "version_date": "2025",
        "ingestion_scope": "full_document",
    },

    "LPI.pdf": {
        "source": "World Bank",
        "real_or_synthetic": "real",
        "document_type": "logistics_performance_index",
        "department": "Operations",
        "sensitivity_level": 1,
        "version_date": "2025",
        "ingestion_scope": "full_document",
    },

    "LPI_Methodology.pdf": {
        "source": "World Bank",
        "real_or_synthetic": "real",
        "document_type": "methodology",
        "department": "Operations",
        "sensitivity_level": 1,
        "version_date": "2026",
        "ingestion_scope": "full_document",
    },

    "MSC-MEPC_3-Circ_4-Rev_1.pdf": {
        "source": "IMO",
        "real_or_synthetic": "real",
        "document_type": "marine_casualty_reporting_guidance",
        "department": "Operations",
        "sensitivity_level": 1,
        "version_date": "2014-11-18",
        "ingestion_scope": "full_document",
    },

    "Transit_Dues_Rates.pdf": {
        "source": "Suez Canal Authority",
        "real_or_synthetic": "real",
        "document_type": "suez_transit_dues_schedule",
        "department": "Finance",
        "sensitivity_level": 1,
        "version_date": "2023-01-01",
        "ingestion_scope": "notes_only",
    },
}


# Build classification records
classification = []

for file in pdf_files:
    filename = file.name

    if filename in core_classification:
        metadata = core_classification[filename].copy()

    elif "us_coast_guard_reports" in str(file):
        metadata = {
            "source": "US Coast Guard",
            "real_or_synthetic": "real",
            "document_type": "marine_casualty_report",
            "department": "Operations",
            "sensitivity_level": 1,
            "version_date": None,
            "ingestion_scope": "full_document",
        }

    else:
        metadata = {
            "source": None,
            "real_or_synthetic": None,
            "document_type": None,
            "department": None,
            "sensitivity_level": None,
            "version_date": None,
            "ingestion_scope": None,
        }

    classification.append({
        "filename": filename,
        **metadata
    })


classification_df = pd.DataFrame(classification)

classification_df

,filename,source,real_or_synthetic,document_type,department,sensitivity_level,version_date,ingestion_scope
0,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,NaN,full_document
1,HbR_Annual_report_2025_highlights.pdf,Port of Rotterdam Authority,real,annual_report,Operations,1,2025,full_document
2,LPI.pdf,World Bank,real,logistics_performance_index,Operations,1,2025,full_document
3,LPI_Methodology.pdf,World Bank,real,methodology,Operations,1,2026,full_document
4,MSC-MEPC_3-Circ_4-Rev_1.pdf,IMO,real,marine_casualty_reporting_guidance,Operations,1,2014-11-18,full_document
5,Transit_Dues_Rates.pdf,Suez Canal Authority,real,suez_transit_dues_schedule,Finance,1,2023-01-01,notes_only
6,BIG_STICK_7895450_ROI_Redacted.pdf,US Coast Guard,real,marine_casualty_report,Operations,1,NaN,full_document
7,CAROL_ANN_7816410_ROI_Redacted.pdf,US Coast Guard,real,marine_casualty_report,Operations,1,NaN,full_document
8,CONCHRD_7107811_ROI_Redacted.pdf,US Coast Guard,real,marine_casualty_report,Operations,1,NaN,full_document
9,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,US Coast Guard,real,marine_casualty_report,Operations,1,NaN,full_document


## 1.3 Classification Validation

The classification must cover every PDF in the inventory.

Expected:
- 30 PDF files
- 6 core RAG documents
- 24 USCG reports
- no unclassified documents

In [5]:
print("Inventory files:", len(inventory_df))
print("Classified files:", len(classification_df))

print("\nMissing classifications:")
print(classification_df[
    classification_df["source"].isna()
])

print("\nDocument type distribution:")
print(classification_df["document_type"].value_counts())

print("\nUSCG reports:")
print(
    (classification_df["source"] == "US Coast Guard").sum()
)

Inventory files: 30
Classified files: 30

Missing classifications:
Empty DataFrame
Columns: [filename, source, real_or_synthetic, document_type, department, sensitivity_level, version_date, ingestion_scope]
Index: []

Document type distribution:
document_type
marine_casualty_report                24
bill_of_lading_template                1
annual_report                          1
logistics_performance_index            1
methodology                            1
marine_casualty_reporting_guidance     1
suez_transit_dues_schedule             1
Name: count, dtype: int64

USCG reports:
24


## 2. PDF Metadata Inspection

Inspect the embedded metadata of each PDF before extracting the document
content.

This step helps identify document-level information such as title, author,
creator, producer, creation date, and modification date.

In [6]:
pdf_metadata = []

for file in pdf_files:
    reader = pypdf.PdfReader(str(file))
    metadata = reader.metadata

    pdf_metadata.append({
        "filename": file.name,
        "title": metadata.get("/Title"),
        "author": metadata.get("/Author"),
        "subject": metadata.get("/Subject"),
        "creator": metadata.get("/Creator"),
        "producer": metadata.get("/Producer"),
        "creation_date": metadata.get("/CreationDate"),
        "modification_date": metadata.get("/ModDate"),
    })

pdf_metadata_df = pd.DataFrame(pdf_metadata)

pdf_metadata_df

,filename,title,author,subject,creator,producer,creation_date,modification_date
0,FBL_template.pdf,NaN,NaN,NaN,NaN,iLovePDF,NaN,D:20260911171838Z
1,HbR_Annual_report_2025_highlights.pdf,Havenbedrijf Rotterdam Jaarverslag 2025,,Jaarverslag 2025,PDFreactor 12.4.0 (Licensee: F19 IP+C #4106),PDFreactor 12.4.0 (Licensee: F19 IP+C #4106),D:20260311124537+01'00',D:20260311124537+01'00'
2,LPI.pdf,NaN,NaN,NaN,Aspose Pty Ltd.,Aspose.PDF for Java 25.8,D:20260422111837-04'00',D:20260422145205-04'00'
3,LPI_Methodology.pdf,NaN,NaN,NaN,Aspose Pty Ltd.,Aspose.PDF for Java 25.8,D:20260407233407-06'00',D:20260422145227-04'00'
4,MSC-MEPC_3-Circ_4-Rev_1.pdf,Session,Carlos Felipe Ormaechea,NaN,Microsoft® Word 2013,Microsoft® Word 2013,D:20141119163222Z,D:20141119163243Z
5,Transit_Dues_Rates.pdf,NaN,NaN,NaN,Adobe Acrobat 8.0 Combine Files,Microsoft: Print To PDF,D:20220918121752+02'00',D:20230125134153+02'00'
6,BIG_STICK_7895450_ROI_Redacted.pdf,U.S. Coast Guard Report of Investigation: Man ...,U.S. Coast Guard,U.S. Coast Guard Report of Investigation: Man ...,Acrobat PDFMaker 25 for Word,Acrobat Distiller 25.0 (Windows),D:20260318125511-04'00',D:20260420131641-04'00'
7,CAROL_ANN_7816410_ROI_Redacted.pdf,U.S. Coast Guard Report of Investigation: Sink...,U.S. Coast Guard,U.S. Coast Guard Report of Investigation: Sink...,NaN,Microsoft: Print To PDF,D:20240910132203-04'00',D:20260209064647-05'00'
8,CONCHRD_7107811_ROI_Redacted.pdf,U.S. Coast Guard Report of Investigation: Caps...,U.S. Coast Guard,U.S. Coast Guard Report of Investigation: Caps...,Acrobat PDFMaker 25 for Word,Adobe PDF Library 25.1.201,D:20251205130205-05'00',D:20260625075508-04'00'
9,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,U.S. Coast Guard Board of Inquiry Report: Port...,U.S. Coast Guard,U.S. Coast Guard Board of Inquiry Report: Port...,NaN,NaN,D:20260903114041-04'00',D:20260903114041-04'00'


In [7]:
pdf_metadata_df.isna().sum()

filename              0
title                 4
author                4
subject               5
creator              17
producer             11
creation_date         1
modification_date     0
dtype: int64

## 3. Text Extraction

Extract text from each PDF page by page.

Keeping the page number is important because it allows the RAG system
to trace each retrieved chunk back to its original document location.

In [8]:
extracted_documents = []

for file in pdf_files:
    reader = pypdf.PdfReader(str(file))

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        extracted_documents.append({
            "filename": file.name,
            "page": page_number,
            "text": text
        })

extracted_df = pd.DataFrame(extracted_documents)

extracted_df.head()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/MAAQDX+HelveticaNeue-Roman', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(16, 0, 2622510626000), '/LastChar': 169, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 259, 259, 0, 0, 278, 389, 278, 333, 0, 556, 556, 0, 556, 0, 556, 556, 556, 556, 278, 0, 0, 0, 0, 0, 0, 648, 685, 722, 704, 611, 574, 759, 0, 259, 0, 0, 556, 871, 722, 760, 648, 0, 685, 648, 574, 722, 0, 926, 0, 0, 611, 0, 0, 0, 0, 0, 0, 537, 593, 537, 593, 537, 296, 574, 556, 222, 222, 519, 222, 853, 556, 574, 593, 0, 333, 500, 315, 556, 500, 758, 518, 500, 480, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 800]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Typ

,filename,page,text
0,FBL_template.pdf,1,Consignor\nFBL\nNEGOTIABLE FIATA\nMULTIMODAL T...
1,FBL_template.pdf,2,Standard Conditions (1992) governing\nthe Nego...
2,FBL_template.pdf,3,Standard Conditions (1992) governing\nthe Nego...
3,HbR_Annual_report_2025_highlights.pdf,1,
4,HbR_Annual_report_2025_highlights.pdf,2,IN THIS \nREPORT\nForeword Executive Board 2\...


### 3.1 Text Length Diagnostics

Inspect the character length of every extracted page before deciding how to handle short or empty pages. Unusually short pages are a signal to check manually; zero-length pages are handled explicitly in 3.2 below.

In [9]:
print("Total extracted pages:", len(extracted_df))
print("Pages with text:", extracted_df["text"].notna().sum())
print("Pages without text:", extracted_df["text"].isna().sum())

Total extracted pages: 820
Pages with text: 820
Pages without text: 0


In [10]:
extracted_df["text_length"] = extracted_df["text"].str.len()

extracted_df["text_length"].describe()

count      820.000000
mean      1918.859756
std       1682.744611
min          0.000000
25%        804.000000
50%       1904.000000
75%       2910.750000
max      24909.000000
Name: text_length, dtype: float64

In [11]:
extracted_df.sort_values("text_length").head(10)

,filename,page,text,text_length
589,PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf,16,,0
590,PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf,17,,0
591,PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf,18,,0
592,PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf,19,,0
593,PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf,20,,0
370,CUYAHOGA_7881082_ROI_Redacted.pdf,4,,0
371,CUYAHOGA_7881082_ROI_Redacted.pdf,5,,0
377,CUYAHOGA_7881082_ROI_Redacted.pdf,11,,0
809,WILDERNESS_I_7828267_ROI_Redacted.pdf,9,,0
770,WAYMON_BOYD_7039383_ROI_Redacted.pdf,8,,0


In [12]:
zero_text_pages = extracted_df[extracted_df["text_length"] == 0]

zero_text_pages[["filename", "page", "text_length"]]

,filename,page,text_length
3,HbR_Annual_report_2025_highlights.pdf,1,0
87,HbR_Annual_report_2025_highlights.pdf,85,0
96,LPI.pdf,9,0
119,LPI_Methodology.pdf,2,0
149,LPI_Methodology.pdf,32,0
...,...,...,...
808,WILDERNESS_I_7828267_ROI_Redacted.pdf,8,0
809,WILDERNESS_I_7828267_ROI_Redacted.pdf,9,0
810,WILDERNESS_I_7828267_ROI_Redacted.pdf,10,0
811,WILDERNESS_I_7828267_ROI_Redacted.pdf,11,0


In [13]:
zero_text_pages["filename"].value_counts()

filename
PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf                  20
QUEEN_OF_THE_WEST_4405289_ROI_Redacted.pdf                   19
H20_BELOW_6966090_ROI_Redacted.pdf                           14
UNA_MAS_8231938_ROI_Redacted.pdf                             11
LITTLE_G_7717011_ROI_Redacted.pdf                             8
WAYMON_BOYD_7039383_ROI_Redacted.pdf                          7
CUYAHOGA_7881082_ROI_Redacted.pdf                             6
LADY_ELLEN_7965695_ROI_Redacted.pdf                           5
WILDERNESS_I_7828267_ROI_Redacted.pdf                         5
KIM_LY_7867677_ROI_Redacted.pdf                               4
SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf                     4
Critical_Port_Infrastructure_Risks_PWSA_BOIR_Redacted.pdf     3
HbR_Annual_report_2025_highlights.pdf                         2
LPI_Methodology.pdf                                           2
CUMBERLAND_II_8126294_ROI_Redacted.pdf                        2
GENERAL_MACARTHUR_7861135_ROI_R

In [14]:
zero_text_pages = extracted_df[extracted_df["text_length"] == 0].copy()

zero_text_pages["filename"].value_counts().sort_values(ascending=False)

filename
PROWLER_ATTESSA_IV_6572770_ROI_Redacted.pdf                  20
QUEEN_OF_THE_WEST_4405289_ROI_Redacted.pdf                   19
H20_BELOW_6966090_ROI_Redacted.pdf                           14
UNA_MAS_8231938_ROI_Redacted.pdf                             11
LITTLE_G_7717011_ROI_Redacted.pdf                             8
WAYMON_BOYD_7039383_ROI_Redacted.pdf                          7
CUYAHOGA_7881082_ROI_Redacted.pdf                             6
LADY_ELLEN_7965695_ROI_Redacted.pdf                           5
WILDERNESS_I_7828267_ROI_Redacted.pdf                         5
KIM_LY_7867677_ROI_Redacted.pdf                               4
SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf                     4
Critical_Port_Infrastructure_Risks_PWSA_BOIR_Redacted.pdf     3
HbR_Annual_report_2025_highlights.pdf                         2
LPI_Methodology.pdf                                           2
CUMBERLAND_II_8126294_ROI_Redacted.pdf                        2
GENERAL_MACARTHUR_7861135_ROI_R

## 3.2 Zero-Text Page Inspection

Inspect zero-text pages to determine whether they contain embedded images.
This helps distinguish image-only pages from pages where text extraction failed.

In [15]:
zero_text_inspection = []

for file in pdf_files:
    reader = pypdf.PdfReader(str(file))

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        if len(text.strip()) == 0:
            images = page.images

            zero_text_inspection.append({
                "filename": file.name,
                "page": page_number,
                "image_count": len(images)
            })

zero_text_df = pd.DataFrame(zero_text_inspection)

zero_text_df.head(20)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/MAAQDX+HelveticaNeue-Roman', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(16, 0, 2622510625520), '/LastChar': 169, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [278, 0, 0, 0, 0, 0, 0, 0, 259, 259, 0, 0, 278, 389, 278, 333, 0, 556, 556, 0, 556, 0, 556, 556, 556, 556, 278, 0, 0, 0, 0, 0, 0, 648, 685, 722, 704, 611, 574, 759, 0, 259, 0, 0, 556, 871, 722, 760, 648, 0, 685, 648, 574, 722, 0, 926, 0, 0, 611, 0, 0, 0, 0, 0, 0, 537, 593, 537, 593, 537, 296, 574, 556, 222, 222, 519, 222, 853, 556, 574, 593, 0, 333, 500, 315, 556, 500, 758, 518, 500, 480, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 278, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 800]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Typ

,filename,page,image_count
0,HbR_Annual_report_2025_highlights.pdf,1,2
1,HbR_Annual_report_2025_highlights.pdf,85,2
2,LPI.pdf,9,0
3,LPI_Methodology.pdf,2,0
4,LPI_Methodology.pdf,32,0
5,MSC-MEPC_3-Circ_4-Rev_1.pdf,4,0
6,CONCHRD_7107811_ROI_Redacted.pdf,4,3
7,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,11,1
8,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,12,1
9,Critical_Port_Infrastructure_Risks_PWSA_BOIR_R...,21,1


In [16]:
zero_text_df["image_count"].value_counts().sort_index()

image_count
0     12
1     90
2      7
3      5
4      2
6      3
11     1
14     1
20     1
21     1
28     1
Name: count, dtype: int64

In [17]:
zero_text_no_image = zero_text_df[
    zero_text_df["image_count"] == 0
]

zero_text_no_image

,filename,page,image_count
2,LPI.pdf,9,0
3,LPI_Methodology.pdf,2,0
4,LPI_Methodology.pdf,32,0
5,MSC-MEPC_3-Circ_4-Rev_1.pdf,4,0
95,SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf,7,0
96,SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf,8,0
97,SPIRIT_OF_BOSTON_7656694_ROI_Redacted.pdf,9,0
104,UNA_MAS_8231938_ROI_Redacted.pdf,9,0
106,UNA_MAS_8231938_ROI_Redacted.pdf,11,0
107,UNA_MAS_8231938_ROI_Redacted.pdf,12,0


## 4. FBL Document Processing

The FBL requires document-specific processing because its PDF structure is different from a standard text document.

The document contains a form page, multi-column content, definitions, numbered clauses, subclauses, and explanatory sections. Therefore, simple page-level text extraction is not sufficient.

The FBL is reconstructed and cleaned first, then divided into logical units before chunking.

### 4.1 Load and Reconstruct the FBL Text

Load the FBL PDF and reconstruct clean, page-aware text. This step only repairs PDF-extraction artifacts (broken characters, spacing); it does not change any wording, so the reconstructed text stays source-faithful.

In [18]:
fbl_path = r"C:\Users\terbo\Desktop\Project\Data\raw\unstructured\FBL_template.pdf"

In [19]:
import fitz
import re



def clean_text(text):
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = text.replace("\x1f", "[ ]")

    # Fix words split across PDF line breaks
    text = re.sub(r"(\w+)-\s*\n\s*([a-z])", r"\1\2", text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def get_blocks(page):
    blocks = []

    for block_no, block in enumerate(page.get_text("blocks")):
        x0, y0, x1, y1, text, *_ = block

        text = text.strip()

        if not text:
            continue

        blocks.append({
            "block_no": block_no,
            "x0": x0,
            "y0": y0,
            "x1": x1,
            "y1": y1,
            "text": text
        })

    return blocks


def is_artifact(text):
    t = text.lower().strip()

    return (
        "text authorized by fiata" in t
        or "copyright reserved" in t
        or "fiata fbl black_pantone" in t
        or re.search(r"\bseite\s+\d+\b", t) is not None
    )


def join_clause_headers(text):
    """
    Page 2:
        1.
        Applicability

    ->
        1. Applicability
    """

    lines = text.splitlines()
    output = []

    i = 0

    while i < len(lines):

        current = lines[i].strip()

        if re.fullmatch(r"\d+\.", current):

            j = i + 1

            while j < len(lines) and not lines[j].strip():
                j += 1

            if j < len(lines):
                output.append(
                    f"{current} {lines[j].strip()}"
                )
                i = j + 1
                continue

        output.append(lines[i])
        i += 1

    return "\n".join(output)


def reconstruct_page_1(page):
    """
    Page 1 is a form, so reconstruct it using the actual
    coordinates of the fields.
    """

    blocks = [
        b for b in get_blocks(page)
        if not is_artifact(b["text"])
    ]

    # ---------------------------------------------------------
    # Header
    # ---------------------------------------------------------

    output = []

    # Block 0: Consignor + FBL
    header_block = next(
        (
            b for b in blocks
            if "Consignor" in b["text"]
            and "FBL" in b["text"]
        ),
        None
    )

    if header_block:
        header_lines = [
            x.strip()
            for x in header_block["text"].splitlines()
            if x.strip()
        ]

        for line in header_lines:
            output.append(line)

    # Emblem block
    emblem_block = next(
        (
            b for b in blocks
            if "Emblem" in b["text"]
            and "Association" in b["text"]
        ),
        None
    )

    if emblem_block:
        output.append(
            "Emblem of National Association"
        )

    # Title block
    title_block = next(
        (
            b for b in blocks
            if "NEGOTIABLE FIATA" in b["text"]
            and "MULTIMODAL TRANSPORT" in b["text"]
            and "BILL OF LADING" in b["text"]
        ),
        None
    )

    if title_block:
        title_lines = [
            x.strip()
            for x in title_block["text"].splitlines()
            if x.strip()
        ]

        # Merge title fragments
        title_parts = []

        for line in title_lines:
            if line.startswith("issued subject to"):
                continue
            title_parts.append(line)

        if title_parts:
            output.append(
                " ".join(title_parts)
            )

        issued_lines = [
            x.strip()
            for x in title_lines
            if x.startswith("issued subject to")
            or (
                "Multimodal Transport Documents"
                in x
            )
        ]

        if issued_lines:
            output.append(
                " ".join(issued_lines)
            )

    output.append("")

    # ---------------------------------------------------------
    # Top fields
    # ---------------------------------------------------------

    field_blocks = {
        "Consigned to order of": None,
        "Notify address": None,
        "Place of receipt": None,
        "Ocean vessel": None,
        "Port of loading": None,
        "Port of discharge": None,
        "Place of delivery": None,
    }

    for b in blocks:

        text = b["text"].strip()

        for field in field_blocks:

            if text == field:
                field_blocks[field] = b

    if field_blocks["Consigned to order of"]:
        output.append("Consigned to order of")

    if field_blocks["Notify address"]:
        output.append("Notify address")

    output.append("")

    if field_blocks["Place of receipt"]:
        output.append("Place of receipt")

    row = []

    if field_blocks["Ocean vessel"]:
        row.append("Ocean vessel")

    if field_blocks["Port of loading"]:
        row.append("Port of loading")

    if row:
        output.append(" | ".join(row))

    row = []

    if field_blocks["Port of discharge"]:
        row.append("Port of discharge")

    if field_blocks["Place of delivery"]:
        row.append("Place of delivery")

    if row:
        output.append(" | ".join(row))

    # ---------------------------------------------------------
    # Cargo fields
    # ---------------------------------------------------------

    cargo_block = next(
        (
            b for b in blocks
            if "Marks and numbers" in b["text"]
            and "Number and kind of packages" in b["text"]
            and "Description of goods" in b["text"]
            and "Gross weight" in b["text"]
            and "Measurement" in b["text"]
        ),
        None
    )

    if cargo_block:

        cargo_lines = [
            x.strip()
            for x in cargo_block["text"].splitlines()
            if x.strip()
        ]

        # The PDF extracted all five fields into one block.
        # They are actually five horizontal fields on the form.
        cargo_labels = [
            "Marks and numbers",
            "Number and kind of packages",
            "Description of goods",
            "Gross weight",
            "Measurement"
        ]

        cargo = []

        for label in cargo_labels:
            if any(label in line for line in cargo_lines):
                cargo.append(label)

        if cargo:
            output.append("")
            output.append(
                " | ".join(cargo)
            )

    # ---------------------------------------------------------
    # Declaration section
    # ---------------------------------------------------------

    declaration_reference = next(
        (
            b for b in blocks
            if b["text"].strip()
            == "according to the declaration of the consignor"
        ),
        None
    )

    declaration_interest = next(
        (
            b for b in blocks
            if "Declaration of Interest of the consignor"
            in b["text"]
        ),
        None
    )

    declared_value = next(
        (
            b for b in blocks
            if "Declared value for ad valorem rate"
            in b["text"]
        ),
        None
    )

    output.append("")

    if declaration_reference:
        output.append(
            "according to the declaration of the consignor"
        )

    if declaration_interest:
        output.append(
            clean_text(
                declaration_interest["text"]
            )
        )

    if declared_value:
        output.append(
            clean_text(
                declared_value["text"]
            )
        )

    # ---------------------------------------------------------
    # Legal statements
    # ---------------------------------------------------------

    legal_blocks = []

    for b in blocks:

        text = b["text"].strip()

        if (
            text.startswith(
                "The goods and instructions are accepted"
            )
            or text.startswith(
                "Taken in charge in apparent good order"
            )
            or text.startswith(
                "One of these Multimodal Transport Bills of Lading"
            )
        ):
            legal_blocks.append(b)

    legal_blocks.sort(
        key=lambda b: b["y0"]
    )

    output.append("")

    for b in legal_blocks:
        output.append(
            clean_text(b["text"])
        )

    # ---------------------------------------------------------
    # Bottom form
    # ---------------------------------------------------------

    bottom_blocks = []

    bottom_keywords = [
        "Freight amount",
        "Freight payable at",
        "Place and date of issue",
        "Cargo Insurance through the undersigned",
        "Number of Original FBL",
        "Stamp and signature",
        "For delivery of goods please apply to",
    ]

    for b in blocks:

        text = b["text"].strip()

        if any(
            keyword.lower() in text.lower()
            for keyword in bottom_keywords
        ):
            bottom_blocks.append(b)

    bottom_blocks.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    # Cargo insurance is one block and contains the checkboxes.
    # Keep it together.
    for b in bottom_blocks:

        text = clean_text(b["text"])

        # Avoid duplicates caused by matching the same block
        # through multiple keywords.
        if text not in output:
            output.append(text)

    return clean_text(
        "\n".join(output)
    )


def reconstruct_page_2(page):
    """
    Page 2: two-column legal document.
    """

    blocks = [
        b for b in get_blocks(page)
        if not is_artifact(b["text"])
    ]

    midpoint = page.rect.width / 2

    left = []
    right = []

    for b in blocks:

        center_x = (
            b["x0"] + b["x1"]
        ) / 2

        if center_x < midpoint:
            left.append(b)
        else:
            right.append(b)

    left.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    right.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    text = (
        "\n".join(b["text"] for b in left)
        + "\n\n"
        + "\n".join(b["text"] for b in right)
    )

    text = join_clause_headers(text)

    return clean_text(text)


def reconstruct_page_3(page):
    """
    Page 3: two-column document.

    The final checklist is special because the PDF stores:

        1.
        text...

        2.
        text...

    as separate blocks.

    We explicitly attach each number to its corresponding
    content.
    """

    blocks = [
        b for b in get_blocks(page)
        if not is_artifact(b["text"])
    ]

    # Remove printed page number "13" at the bottom.
    blocks = [
        b for b in blocks
        if not (
            b["text"].strip() == "13"
            and b["y0"] > page.rect.height * 0.90
        )
    ]

    # ---------------------------------------------------------
    # Header spanning both columns
    # ---------------------------------------------------------

    header_blocks = [
        b for b in blocks
        if b["x0"] < 100
        and b["x1"] > page.rect.width - 100
        and b["y0"] < 115
    ]

    header_blocks.sort(
        key=lambda b: b["y0"]
    )

    header_text = "\n".join(
        b["text"]
        for b in header_blocks
    )

    # Remove header blocks from columns.
    header_ids = {
        b["block_no"]
        for b in header_blocks
    }

    blocks = [
        b for b in blocks
        if b["block_no"] not in header_ids
    ]

    # ---------------------------------------------------------
    # Split columns
    # ---------------------------------------------------------

    midpoint = page.rect.width / 2

    left = []
    right = []

    for b in blocks:

        center_x = (
            b["x0"] + b["x1"]
        ) / 2

        if center_x < midpoint:
            left.append(b)
        else:
            right.append(b)

    left.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    right.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    # ---------------------------------------------------------
    # Merge the 1–5 checklist numbers with their content.
    # ---------------------------------------------------------

    number_blocks = []

    for b in right:

        if re.fullmatch(
            r"[1-5]\.",
            b["text"].strip()
        ):
            number_blocks.append(b)

    used_numbers = set()
    used_content = set()

    merged_checklist = []

    for number_block in number_blocks:

        number = number_block["text"].strip()

        candidates = [
            b for b in right
            if b["block_no"] not in used_content
            and b["block_no"] != number_block["block_no"]
            and b["x0"] > number_block["x1"]
            and abs(b["y0"] - number_block["y0"]) <= 3
        ]

        if not candidates:
            continue

        content = min(
            candidates,
            key=lambda b: b["x0"]
        )

        merged_checklist.append({
            "y0": number_block["y0"],
            "x0": number_block["x0"],
            "text":
                number
                + " "
                + content["text"]
        })

        used_numbers.add(
            number_block["block_no"]
        )

        used_content.add(
            content["block_no"]
        )

    # ---------------------------------------------------------
    # Rebuild right column.
    # ---------------------------------------------------------

    right_output = []

    for b in right:

        if b["block_no"] in used_numbers:
            continue

        if b["block_no"] in used_content:
            continue

        right_output.append({
            "y0": b["y0"],
            "x0": b["x0"],
            "text": b["text"]
        })

    right_output.extend(merged_checklist)

    right_output.sort(
        key=lambda b: (b["y0"], b["x0"])
    )

    # ---------------------------------------------------------
    # Final Page 3
    # ---------------------------------------------------------

    page_text_parts = []

    if header_text.strip():
        page_text_parts.append(header_text)

    page_text_parts.append(
        "\n".join(
            b["text"]
            for b in left
        )
    )

    page_text_parts.append(
        "\n".join(
            b["text"]
            for b in right_output
        )
    )

    return clean_text(
        "\n\n".join(page_text_parts)
    )


def reconstruct_fbl(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(
        doc,
        start=1
    ):

        if page_number == 1:

            page_text = reconstruct_page_1(page)

        elif page_number == 2:

            page_text = reconstruct_page_2(page)

        elif page_number == 3:

            page_text = reconstruct_page_3(page)

        else:
            page_text = clean_text(
                page.get_text()
            )

        pages.append(
            f"========== PAGE {page_number} ==========\n\n"
            + page_text
        )

    doc.close()

    return "\n\n".join(pages)


# =============================================================
# FINAL RECONSTRUCTION
# =============================================================

fbl_text = reconstruct_fbl(fbl_path)

print(fbl_text)

========== PAGE 1 ==========

Consignor
FBL
NEGOTIABLE FIATA MULTIMODAL TRANSPORT BILL OF LADING Multimodal Transport Documents (ICC Publication 481).
issued subject to UNCTAD / ICC Rules for Multimodal Transport Documents (ICC Publication 481).

Consigned to order of
Notify address

Place of receipt
Ocean vessel | Port of loading
Port of discharge | Place of delivery

Marks and numbers | Number and kind of packages | Description of goods | Gross weight | Measurement

according to the declaration of the consignor
Declaration of Interest of the consignor
in timely delivery (Clause 6.2.)
Declared value for ad valorem rate according to
the declaration of the consignor (Clauses 7 and 8).

The goods and instructions are accepted and dealt with subject to the Standard Conditions printed overleaf.
Taken in charge in apparent good order and condition, unless otherwise noted herein, at the place of receipt for transport and delivery as mentioned above.
One of these Multimodal Transport Bills of

### 4.2 FBL Logical Unit Extraction

After reconstructing the FBL text, the document is divided according to its logical structure.

Page 1 is preserved as the FBL form, while Page 2 is divided into Definitions and Clauses 1–19. Page 3 is represented by its explanatory sections.

The parser also validates that all top-level clauses from 1 to 19 are detected in the expected order.

In [20]:
import re

def parse_fbl_logical_units(fbl_text):

    units = []

    # =========================================================
    # 1. Split FBL into physical pages
    # =========================================================

    page_pattern = re.compile(
        r"={10,}\s*PAGE\s+(\d+)\s*={10,}",
        flags=re.IGNORECASE
    )

    matches = list(page_pattern.finditer(fbl_text))

    if len(matches) != 3:
        raise ValueError(
            f"Expected 3 pages, found {len(matches)}"
        )

    pages = {}

    for i, match in enumerate(matches):

        page_number = int(match.group(1))
        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(fbl_text)

        pages[page_number] = fbl_text[start:end].strip()

    # =========================================================
    # 2. PAGE 1 → FBL Form
    # =========================================================

    units.append({
        "page": 1,
        "unit_type": "form",
        "logical_unit": "fbl_form",
        "clause_number": None,
        "subclause_number": None,
        "title": "Negotiable FIATA Multimodal Transport Bill of Lading",
        "text": pages[1]
    })

    # =========================================================
    # 3. PAGE 2 → Definitions + Clauses
    # =========================================================

    page2 = pages[2]

    # ---------------------------------------------------------
    # Find Clause 1
    # ---------------------------------------------------------

    clause1_match = re.search(
        r"(?m)^\s*1\.\s+Applicability\s*$",
        page2
    )

    if not clause1_match:

        # fallback: search without requiring exact line structure
        clause1_match = re.search(
            r"\b1\.\s+Applicability\b",
            page2,
            flags=re.IGNORECASE
        )

    if not clause1_match:
        raise ValueError("Could not find Clause 1 on Page 2")

    # Everything before Clause 1 belongs to Definitions
    definitions_text = page2[:clause1_match.start()].strip()

    # Remove document heading before "Definitions"
    definitions_match = re.search(
        r"\bDefinitions\b",
        definitions_text,
        flags=re.IGNORECASE
    )

    if definitions_match:
        definitions_text = definitions_text[
            definitions_match.start():
        ].strip()

    units.append({
        "page": 2,
        "unit_type": "definitions",
        "logical_unit": "definitions",
        "clause_number": None,
        "subclause_number": None,
        "title": "Definitions",
        "text": definitions_text
    })

    # ---------------------------------------------------------
    # Find all top-level clauses
    # ---------------------------------------------------------

    clause_pattern = re.compile(
        r"(?m)^\s*(\d+)\.\s+([^\n]+?)\s*$"
    )

    clause_matches = [
        m for m in clause_pattern.finditer(page2)
        if 1 <= int(m.group(1)) <= 19
    ]

    # Check that we found exactly Clauses 1–19
    clause_numbers = [int(m.group(1)) for m in clause_matches]

    if clause_numbers != list(range(1, 20)):
        raise ValueError(
            f"Expected clauses 1-19, found: {clause_numbers}"
        )

    # ---------------------------------------------------------
    # Build Clause logical units
    # ---------------------------------------------------------

    for i, match in enumerate(clause_matches):

        clause_number = int(match.group(1))
        clause_title = match.group(2).strip()

        start = match.start()

        if i + 1 < len(clause_matches):
            end = clause_matches[i + 1].start()
        else:
            end = len(page2)

        clause_text = page2[start:end].strip()

        # Remove ICC footer/note from Clause 19
        if clause_number == 19:

            footer_match = re.search(
                r"The ICC logo denotes that this document",
                clause_text,
                flags=re.IGNORECASE
            )

            if footer_match:
                clause_text = clause_text[
                    :footer_match.start()
                ].strip()

        units.append({
            "page": 2,
            "unit_type": "clause",
            "logical_unit": f"clause_{clause_number}",
            "clause_number": clause_number,
            "subclause_number": None,
            "title": clause_title,
            "text": clause_text
        })

    # =========================================================
    # 4. PAGE 3 → Semantic Logical Units
    # =========================================================

    page3 = pages[3]

    # For boundary detection only:
    # normalize whitespace temporarily.
    page3_search = re.sub(
        r"\s+",
        " ",
        page3
    ).strip()

    # ---------------------------------------------------------
    # Locate semantic boundaries
    # ---------------------------------------------------------

    liability_start = page3_search.lower().find(
        "by issuing a fbl"
    )

    issuing_start = page3_search.lower().find(
        "when issuing an fbl"
    )

    insurance_start = page3_search.lower().find(
        "freight forwarders issuing fbls have to insure their liability"
    )

    if liability_start == -1:
        raise ValueError(
            "Could not find Liability Limits section on Page 3"
        )

    if issuing_start == -1:
        raise ValueError(
            "Could not find Issuing Requirements section on Page 3"
        )

    if insurance_start == -1:
        raise ValueError(
            "Could not find Liability Insurance section on Page 3"
        )

    # ---------------------------------------------------------
    # Overview
    # ---------------------------------------------------------

    overview_text = page3_search[:liability_start].strip()

    # Remove document title
    overview_text = re.sub(
        r"^FBL\s+Negotiable FIATA Multimodal Transport Bill of Lading\s*",
        "",
        overview_text,
        flags=re.IGNORECASE
    ).strip()

    units.append({
        "page": 3,
        "unit_type": "explanatory",
        "logical_unit": "fbl_overview",
        "clause_number": None,
        "subclause_number": None,
        "title": "FBL Overview",
        "text": overview_text
    })

    # ---------------------------------------------------------
    # Liability Limits
    # ---------------------------------------------------------

    liability_text = page3_search[
        liability_start:issuing_start
    ].strip()

    units.append({
        "page": 3,
        "unit_type": "explanatory",
        "logical_unit": "liability_limits",
        "clause_number": None,
        "subclause_number": None,
        "title": "FBL Liability Limits",
        "text": liability_text
    })

    # ---------------------------------------------------------
    # Issuing FBL Requirements
    # Keep the entire checklist together
    # ---------------------------------------------------------

    issuing_text = page3_search[
        issuing_start:insurance_start
    ].strip()

    units.append({
        "page": 3,
        "unit_type": "checklist",
        "logical_unit": "issuing_fbl_requirements",
        "clause_number": None,
        "subclause_number": None,
        "title": "Requirements When Issuing an FBL",
        "text": issuing_text
    })

    # ---------------------------------------------------------
    # Liability Insurance
    # ---------------------------------------------------------

    insurance_text = page3_search[
        insurance_start:
    ].strip()

    units.append({
        "page": 3,
        "unit_type": "explanatory",
        "logical_unit": "liability_insurance",
        "clause_number": None,
        "subclause_number": None,
        "title": "Liability Insurance",
        "text": insurance_text
    })

    return units

In [21]:
fbl_units = parse_fbl_logical_units(fbl_text)

In [22]:
for i, unit in enumerate(fbl_units):

    print(
        f"{i:02d} | "
        f"Page {unit['page']} | "
        f"{unit['unit_type']:12} | "
        f"{unit['logical_unit']:30} | "
        f"{len(unit['text']):5} chars | "
        f"{unit['title']}"
    )

00 | Page 1 | form         | fbl_form                       |  1475 chars | Negotiable FIATA Multimodal Transport Bill of Lading
01 | Page 2 | definitions  | definitions                    |   964 chars | Definitions
02 | Page 2 | clause       | clause_1                       |   167 chars | Applicability
03 | Page 2 | clause       | clause_2                       |   711 chars | Issuance of this FBL
04 | Page 2 | clause       | clause_3                       |   804 chars | Negotiability and title to the goods
05 | Page 2 | clause       | clause_4                       |  1550 chars | Dangerous Goods and Indemnity
06 | Page 2 | clause       | clause_5                       |  1570 chars | Description of Goods and Merchant’s Packing and Inspection
07 | Page 2 | clause       | clause_6                       |  3137 chars | Freight Forwarder’s Liability
08 | Page 2 | clause       | clause_7                       |  1055 chars | Paramount clauses
09 | Page 2 | clause       | clause_8     

In [23]:
fbl_units_df = pd.DataFrame(fbl_units)

fbl_units_df["char_count"] = fbl_units_df["text"].str.len()

fbl_units_df[
    [
        "logical_unit",
        "title",
        "char_count"
    ]
].sort_values("char_count", ascending=False)

,logical_unit,title,char_count
9,clause_8,Limitation of Freight Forwarder’s Liability,3570
7,clause_6,Freight Forwarder’s Liability,3137
14,clause_13,Freight and Charges,2529
13,clause_12,Delivery,1609
6,clause_5,Description of Goods and Merchant’s Packing an...,1570
5,clause_4,Dangerous Goods and Indemnity,1550
0,fbl_form,Negotiable FIATA Multimodal Transport Bill of ...,1475
11,clause_10,Liability of Servants and other Persons,1357
21,fbl_overview,FBL Overview,1356
8,clause_7,Paramount clauses,1055


### 4.3 Identifying Large Logical Units

The extracted logical units are inspected by character count to identify units that may be too large for a single retrieval chunk.

A threshold of 2000 characters is used for this inspection. Large clauses are then examined at the subclause level so that any required splitting follows the document's existing structure.

In [24]:
large_units = [
    unit for unit in fbl_units
    if len(unit["text"]) > 2000
]

for unit in large_units:
    print("\n" + "=" * 80)
    print(
        unit["logical_unit"],
        "|",
        unit["title"],
        "|",
        len(unit["text"]),
        "chars"
    )
    print("=" * 80)
    print(unit["text"])


clause_6 | Freight Forwarder’s Liability | 3137 chars
6. Freight Forwarder’s Liability
6.1. The responsibility of the Freight Forwarder for the goods under these conditions covers the period
from the time the Freight Forwarder has taken the goods in his charge to the time of their delivery.
6.2. The Freight Forwarder shall be liable for loss of or damage to the goods as well as for delay in delivery
if the occurrence which caused the loss, damage or delay in delivery took place while the goods were
in his charge as defined in Clause 2.1.a, unless the Freight Forwarder proves that no fault or neglect of
his own, his servants or agents or any other person referred to in Clause 2.2. has caused or contributed
to such loss, damage or delay. However, the Freight Forwarder shall only be liable for loss following
from delay in delivery if the Consignor has made a declaration of interest in timely delivery which has
been accepted by the Freight Forwarder and stated in this FBL.
6.3. Arrival ti

In [25]:
large_clauses = [
    unit for unit in fbl_units
    if unit["logical_unit"] in ["clause_6", "clause_8", "clause_13"]
]

for unit in large_clauses:
    print("\n" + "=" * 80)
    print(f"{unit['logical_unit']} | {unit['title']}")
    print("=" * 80)

    # Find subclauses like 6.1., 6.2., 8.1., 13.1., etc.
    pattern = rf"(?m)^\s*{unit['clause_number']}\.(\d+)\.\s+"

    matches = list(re.finditer(pattern, unit["text"]))

    for i, match in enumerate(matches):
        subclause_number = int(match.group(1))

        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(unit["text"])

        subclause_text = unit["text"][start:end].strip()

        print(
            f"{unit['clause_number']}.{subclause_number}"
            f" → {len(subclause_text)} chars"
        )


clause_6 | Freight Forwarder’s Liability
6.1 → 204 chars
6.2 → 691 chars
6.3 → 346 chars
6.4 → 221 chars
6.5 → 885 chars
6.6 → 752 chars

clause_8 | Limitation of Freight Forwarder’s Liability
8.1 → 274 chars
8.2 → 271 chars
8.3 → 627 chars
8.4 → 333 chars
8.5 → 325 chars
8.6 → 841 chars
8.7 → 361 chars
8.8 → 123 chars
8.9 → 359 chars

clause_13 | Freight and Charges
13.1 → 324 chars
13.2 → 569 chars
13.3 → 280 chars
13.4 → 277 chars
13.5 → 734 chars
13.6 → 316 chars


## 5. FBL Structure-Aware Chunking

Large FBL clauses are split using their natural subclause boundaries rather than arbitrary character positions.

Clauses 6, 8, and 13 are divided according to predefined groups of subclauses. The parent clause heading is retained in every resulting chunk so that each chunk remains understandable when retrieved independently.

FBL's own structure is expressed in legal clauses and subclauses, but the resulting chunks store that relationship in the generic `parent_chunk` / `sub_chunk` fields of the unified schema (defined in Section 6). Keeping the field names generic lets other resource types — sections, articles, list items, and so on — reuse the same schema without being described as "clauses".

In [26]:
import re


def split_clause_by_subclauses(unit, groups):

    text = unit["text"]
    clause_number = unit["clause_number"]

    pattern = rf"(?m)^\s*{clause_number}\.(\d+)\.\s+"
    matches = list(re.finditer(pattern, text))

    subclauses = {}

    for i, match in enumerate(matches):

        sub_number = match.group(1)

        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(text)

        subclauses[sub_number] = text[start:end].strip()

    chunks = []

    # Extract the main clause heading
    first_subclause_start = matches[0].start()

    clause_heading = text[:first_subclause_start].strip()

    for chunk_index, group in enumerate(groups, start=1):

        chunk_parts = []

        for sub_number in group:

            if sub_number not in subclauses:
                raise ValueError(
                    f"Could not find subclause "
                    f"{clause_number}.{sub_number}"
                )

            chunk_parts.append(subclauses[sub_number])

        # IMPORTANT:
        # Keep the main clause heading inside every chunk
        chunk_text = "\n\n".join(
            [clause_heading] + chunk_parts
        )

        chunks.append({
            "page": unit["page"],
            "unit_type": unit["unit_type"],
            "logical_unit": unit["logical_unit"],
            "chunk_id": f"{unit['logical_unit']}_chunk_{chunk_index}",
            # Generic structural fields: for FBL these hold the clause
            # number and the subclause numbers it was split from, but
            # another resource type could use section IDs or list
            # positions instead.
            "parent_chunk": clause_number,
            "sub_chunk": [
                f"{clause_number}.{x}" for x in group
            ],
            "title": unit["title"],
            "text": chunk_text,
            "char_count": len(chunk_text)
        })

    return chunks


def build_fbl_final_chunks(fbl_units, threshold=2000):

    final_chunks = []

    split_plan = {
        6: [
            ["1", "2", "3", "4"],
            ["5"],
            ["6"]
        ],
        8: [
            ["1", "2", "3"],
            ["4", "5", "6"],
            ["7", "8", "9"]
        ],
        13: [
            ["1", "2", "3", "4"],
            ["5"],
            ["6"]
        ]
    }

    for unit in fbl_units:

        # -----------------------------------------------------
        # Normal logical unit
        # -----------------------------------------------------

        if len(unit["text"]) <= threshold:

            final_chunks.append({
                "page": unit["page"],
                "unit_type": unit["unit_type"],
                "logical_unit": unit["logical_unit"],
                "chunk_id": f"{unit['logical_unit']}_chunk_1",
                "parent_chunk": unit["clause_number"],
                "sub_chunk": None,
                "title": unit["title"],
                "text": unit["text"],
                "char_count": len(unit["text"])
            })

        # -----------------------------------------------------
        # Large logical unit → split according to structure
        # -----------------------------------------------------

        else:

            clause_number = unit["clause_number"]

            if clause_number not in split_plan:
                raise ValueError(
                    f"Large unit '{unit['logical_unit']}' "
                    f"has no defined split plan."
                )

            chunks = split_clause_by_subclauses(
                unit,
                split_plan[clause_number]
            )

            final_chunks.extend(chunks)

    return final_chunks


In [27]:
fbl_chunks = build_fbl_final_chunks(fbl_units)

In [28]:
fbl_chunks_df = pd.DataFrame(fbl_chunks)

print("Total final chunks:", len(fbl_chunks_df))

Total final chunks: 31


In [29]:
fbl_chunks_df[
    [
        "chunk_id",
        "logical_unit",
        "sub_chunk",
        "char_count",
        "title"
    ]
]


,chunk_id,logical_unit,sub_chunk,char_count,title
0,fbl_form_chunk_1,fbl_form,None,1475,Negotiable FIATA Multimodal Transport Bill of ...
1,definitions_chunk_1,definitions,None,964,Definitions
2,clause_1_chunk_1,clause_1,None,167,Applicability
3,clause_2_chunk_1,clause_2,None,711,Issuance of this FBL
4,clause_3_chunk_1,clause_3,None,804,Negotiability and title to the goods
5,clause_4_chunk_1,clause_4,None,1550,Dangerous Goods and Indemnity
6,clause_5_chunk_1,clause_5,None,1570,Description of Goods and Merchant’s Packing an...
7,clause_6_chunk_1,clause_6,"[6.1, 6.2, 6.3, 6.4]",1502,Freight Forwarder’s Liability
8,clause_6_chunk_2,clause_6,[6.5],919,Freight Forwarder’s Liability
9,clause_6_chunk_3,clause_6,[6.6],786,Freight Forwarder’s Liability


### 5.1 FBL Chunk Validation

The final FBL chunks are validated for size, empty content, duplicate identifiers, and overall chunk distribution.

Additional validation checks that the original subclauses remain fully preserved after splitting.

In [30]:
# =========================================================
# Validate final FBL chunks
# =========================================================

print("Total chunks:", len(fbl_chunks_df))

# 1. Check maximum chunk size
print(
    "Maximum chunk size:",
    fbl_chunks_df["char_count"].max()
)

# 2. Check minimum chunk size
print(
    "Minimum chunk size:",
    fbl_chunks_df["char_count"].min()
)

# 3. Check chunks above threshold
print(
    "Chunks above 2000 chars:",
    (fbl_chunks_df["char_count"] > 2000).sum()
)

# 4. Check empty chunks
print(
    "Empty chunks:",
    (fbl_chunks_df["text"].str.strip() == "").sum()
)

# 5. Check duplicated chunk IDs
print(
    "Duplicate chunk IDs:",
    fbl_chunks_df["chunk_id"].duplicated().sum()
)

Total chunks: 31
Maximum chunk size: 1609
Minimum chunk size: 101
Chunks above 2000 chars: 0
Empty chunks: 0
Duplicate chunk IDs: 0


In [31]:
# =========================================================
# Content preservation validation
# =========================================================

original_chars = sum(
    len(unit["text"])
    for unit in fbl_units
)

chunked_chars = sum(
    len(chunk["text"])
    for chunk in fbl_chunks
)

print("Original logical-unit characters:", original_chars)
print("Final chunk characters:", chunked_chars)
print("Difference:", original_chars - chunked_chars)

Original logical-unit characters: 25709
Final chunk characters: 25931
Difference: -222


In [32]:
# =========================================================
# Validate that every subclause is preserved
# =========================================================

for clause_number in [6, 8, 13]:

    original_unit = next(
        unit for unit in fbl_units
        if unit["clause_number"] == clause_number
    )

    original_text = original_unit["text"]

    pattern = rf"(?m)^\s*{clause_number}\.(\d+)\.\s+"
    matches = list(re.finditer(pattern, original_text))

    print("\n" + "=" * 70)
    print(f"Clause {clause_number}")
    print("=" * 70)

    for i, match in enumerate(matches):

        sub_number = match.group(1)
        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(original_text)

        subclause_text = original_text[start:end].strip()

        # Find the corresponding final chunk
        matching_chunks = [
            chunk
            for chunk in fbl_chunks
            if f"{clause_number}.{sub_number}"
            in (chunk["sub_chunk"] or [])
        ]

        preserved = any(
            subclause_text in chunk["text"]
            for chunk in matching_chunks
        )

        print(
            f"{clause_number}.{sub_number} "
            f"→ preserved: {preserved}"
        )



Clause 6
6.1 → preserved: True
6.2 → preserved: True
6.3 → preserved: True
6.4 → preserved: True
6.5 → preserved: True
6.6 → preserved: True

Clause 8
8.1 → preserved: True
8.2 → preserved: True
8.3 → preserved: True
8.4 → preserved: True
8.5 → preserved: True
8.6 → preserved: True
8.7 → preserved: True
8.8 → preserved: True
8.9 → preserved: True

Clause 13
13.1 → preserved: True
13.2 → preserved: True
13.3 → preserved: True
13.4 → preserved: True
13.5 → preserved: True
13.6 → preserved: True


In [33]:
# Final FBL chunking validation

print("Total chunks:", len(fbl_chunks_df))

print("\nMissing text:")
print(fbl_chunks_df["text"].isna().sum())

print("\nEmpty chunks:")
print((fbl_chunks_df["text"].str.strip() == "").sum())

print("\nChunk size statistics:")
print(fbl_chunks_df["char_count"].describe())

print("\nDuplicate chunks:")
print(
    fbl_chunks_df["text"]
    .duplicated()
    .sum()
)

print("\nChunk distribution by logical unit:")
print(
    fbl_chunks_df["logical_unit"]
    .value_counts()
    .sort_index()
)

print("\nOversized chunks (> 2000 chars):")
print(
    fbl_chunks_df[
        fbl_chunks_df["char_count"] > 2000
    ][
        ["chunk_id", "logical_unit", "char_count"]
    ]
)

Total chunks: 31

Missing text:
0

Empty chunks:
0

Chunk size statistics:
count      31.000000
mean      836.483871
std       511.936186
min       101.000000
25%       321.500000
50%       786.000000
75%      1356.500000
max      1609.000000
Name: char_count, dtype: float64

Duplicate chunks:
0

Chunk distribution by logical unit:
logical_unit
clause_1                    1
clause_10                   1
clause_11                   1
clause_12                   1
clause_13                   3
clause_14                   1
clause_15                   1
clause_16                   1
clause_17                   1
clause_18                   1
clause_19                   1
clause_2                    1
clause_3                    1
clause_4                    1
clause_5                    1
clause_6                    3
clause_7                    1
clause_8                    3
clause_9                    1
definitions                 1
fbl_form                    1
fbl_overview           

## 6. FBL Chunk Metadata

Document-level metadata is attached to every FBL chunk to preserve its identity and access-control attributes.

The metadata includes the filename, source, document type, department, sensitivity level, and version information, together with the structural fields already generated during chunking.

In [34]:
# =========================================================
# Canonical flexible chunk schema
# =========================================================
# Structural fields (page, parent_chunk, sub_chunk, title) are nullable
# when a resource does not have pages, a parent/sub-chunk relationship,
# or a natural title. parent_chunk / sub_chunk are intentionally generic:
# for FBL they hold a clause number and its subclause numbers, but another
# resource type could use section IDs, heading paths, or list positions
# instead. This keeps one retrieval schema across resource types without
# changing chunk text.
CHUNK_SCHEMA = {
    "page": "Int64",
    "unit_type": "string",
    "logical_unit": "string",
    "chunk_id": "string",
    "parent_chunk": "string",
    "sub_chunk": "object",
    "title": "string",
    "text": "string",
    "char_count": "Int64",
    "filename": "string",
    "source": "string",
    "real_or_synthetic": "string",
    "document_type": "string",
    "department": "string",
    "sensitivity_level": "Int64",
    "version_date": "string",
}

REQUIRED_CHUNK_FIELDS = [
    "unit_type", "logical_unit", "chunk_id", "text",
    "char_count", "filename", "source", "real_or_synthetic",
    "document_type", "department", "sensitivity_level",
]

OPTIONAL_CHUNK_DEFAULTS = {
    "page": pd.NA,
    "parent_chunk": pd.NA,
    "sub_chunk": None,
    "title": pd.NA,
    "version_date": pd.NA,
}


def normalize_chunk_schema(chunks_df):
    """Validate, order, and normalize nullable types without altering chunk values."""

    expected_columns = list(CHUNK_SCHEMA)
    missing_columns = [
        column for column in REQUIRED_CHUNK_FIELDS
        if column not in chunks_df.columns
    ]
    unexpected_columns = [
        column for column in chunks_df.columns
        if column not in CHUNK_SCHEMA
    ]

    if missing_columns or unexpected_columns:
        raise ValueError(
            f"Schema mismatch. Missing: {missing_columns}; "
            f"Unexpected: {unexpected_columns}"
        )

    normalized = chunks_df.copy()

    for column, default_value in OPTIONAL_CHUNK_DEFAULTS.items():
        if column not in normalized.columns:
            normalized[column] = default_value

    return normalized.loc[:, expected_columns].astype(CHUNK_SCHEMA)


def validate_chunk_schema(chunks_df, label):
    """
    Run the canonical schema and content checks on a chunks DataFrame.

    Checks:
    - columns match CHUNK_SCHEMA, in order
    - dtypes match CHUNK_SCHEMA
    - every required field (REQUIRED_CHUNK_FIELDS) is fully populated
    - chunk_id values are unique
    - text is non-empty and char_count matches the text length
    - sensitivity_level is a valid RBAC level (1, 2, or 3)
    - sub_chunk is only populated when parent_chunk is also populated

    Prints a PASS/FAIL report for every check, then raises an
    AssertionError naming the failed checks if any check failed.
    Returns the dict of individual check results otherwise.
    """

    expected_columns = list(CHUNK_SCHEMA)
    expected_dtypes = pd.Series(CHUNK_SCHEMA).astype(str)

    has_sub_chunk = chunks_df["sub_chunk"].apply(
        lambda value: isinstance(value, (list, tuple)) and len(value) > 0
    )
    has_parent_chunk = chunks_df["parent_chunk"].notna()

    checks = {
        "columns_match_schema": list(chunks_df.columns) == expected_columns,
        "dtypes_match_schema": chunks_df.dtypes.astype(str).equals(expected_dtypes),
        "required_fields_complete": not chunks_df[REQUIRED_CHUNK_FIELDS].isna().any().any(),
        "chunk_id_unique": chunks_df["chunk_id"].is_unique,
        "text_non_empty": chunks_df["text"].str.strip().ne("").all(),
        "char_count_matches_text_length": (
            chunks_df["text"].str.len() == chunks_df["char_count"]
        ).all(),
        "sensitivity_level_in_range": chunks_df["sensitivity_level"].isin([1, 2, 3]).all(),
        "sub_chunk_requires_parent_chunk": (~has_sub_chunk | has_parent_chunk).all(),
    }

    failed = [name for name, passed in checks.items() if not passed]

    print(f"--- Schema validation: {label} ---")
    for name, passed in checks.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {name}")

    if failed:
        raise AssertionError(f"{label}: failed checks -> {failed}")

    print(f"{label}: all checks passed ({len(chunks_df)} chunks)\n")
    return checks


# FBL document metadata

fbl_metadata = {
    "source": "FIATA",
    "real_or_synthetic": "real",
    "document_type": "bill_of_lading_template",
    "department": "Operations",
    "sensitivity_level": 1,
    "version_date": None,
}

# Add filename
fbl_chunks_df["filename"] = "FBL_template.pdf"

# Add document-level metadata
for key, value in fbl_metadata.items():
    fbl_chunks_df[key] = value

# Schema-only refinement: preserve all chunk values and normalize
# their column order and nullable data types.
fbl_chunks_df = normalize_chunk_schema(fbl_chunks_df)

print("Columns:")
print(fbl_chunks_df.columns.tolist())

print("\nShape:")
print(fbl_chunks_df.shape)

print("\nMetadata sample:")
display(
    fbl_chunks_df[
        [
            "filename",
            "source",
            "real_or_synthetic",
            "document_type",
            "department",
            "sensitivity_level",
            "version_date",
            "page",
            "logical_unit",
            "chunk_id",
            "parent_chunk",
            "sub_chunk",
            "title",
            "char_count",
        ]
    ].head()
)


Columns:
['page', 'unit_type', 'logical_unit', 'chunk_id', 'parent_chunk', 'sub_chunk', 'title', 'text', 'char_count', 'filename', 'source', 'real_or_synthetic', 'document_type', 'department', 'sensitivity_level', 'version_date']

Shape:
(31, 16)

Metadata sample:


,filename,source,real_or_synthetic,document_type,department,sensitivity_level,version_date,page,logical_unit,chunk_id,parent_chunk,sub_chunk,title,char_count
0,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>,1,fbl_form,fbl_form_chunk_1,<NA>,None,Negotiable FIATA Multimodal Transport Bill of ...,1475
1,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>,2,definitions,definitions_chunk_1,<NA>,None,Definitions,964
2,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>,2,clause_1,clause_1_chunk_1,1.0,None,Applicability,167
3,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>,2,clause_2,clause_2_chunk_1,2.0,None,Issuance of this FBL,711
4,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>,2,clause_3,clause_3_chunk_1,3.0,None,Negotiability and title to the goods,804


### 6.1 Metadata & Schema Validation

Run the shared schema validator against the FBL chunks — this is the FBL_chunk test — then inspect the document-level metadata that was attached to confirm every chunk carries a single, consistent set of values.

In [35]:
# 6.1 Metadata & schema validation for FBL chunks
validate_chunk_schema(fbl_chunks_df, "FBL_chunks")

required_metadata = [
    "filename",
    "source",
    "real_or_synthetic",
    "document_type",
    "department",
    "sensitivity_level",
    "version_date",
]

print("Unique document-level metadata:")
display(
    fbl_chunks_df[required_metadata]
    .drop_duplicates()
)

print("\nChunk count:", len(fbl_chunks_df))
print("Unique chunk IDs:", fbl_chunks_df["chunk_id"].nunique())


--- Schema validation: FBL_chunks ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] sub_chunk_requires_parent_chunk
FBL_chunks: all checks passed (31 chunks)

Unique document-level metadata:


,filename,source,real_or_synthetic,document_type,department,sensitivity_level,version_date
0,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>



Chunk count: 31
Unique chunk IDs: 31


## 7. Save FBL Processing Artifacts

Persist the reconstructed text, the logical units, and the final chunks so later stages (embeddings, Qdrant ingestion) can reuse them without re-running the FBL-specific extraction and chunking logic above.

In [36]:
# =========================================================
# Save FBL processing artifacts
# =========================================================

from pathlib import Path
import json

# Use the existing project structure
processed_rag_dir = Path(
    r"C:\Users\terbo\Desktop\Project\data\processed\rag"
)

extracted_dir = processed_rag_dir / "extracted"
cleaned_dir = processed_rag_dir / "cleaned"
chunks_dir = processed_rag_dir / "chunks"

# ---------------------------------------------------------
# 1. Save reconstructed/extracted text
# ---------------------------------------------------------

extracted_path = extracted_dir / "FBL_template_reconstructed.txt"

extracted_path.write_text(
    fbl_text,
    encoding="utf-8"
)

# ---------------------------------------------------------
# 2. Save cleaned logical units
# ---------------------------------------------------------

cleaned_path = cleaned_dir / "FBL_template_logical_units.json"

with open(cleaned_path, "w", encoding="utf-8") as f:
    json.dump(
        fbl_units,
        f,
        ensure_ascii=False,
        indent=2
    )

# ---------------------------------------------------------
# 3. Save final chunks
# ---------------------------------------------------------

chunks_parquet_path = chunks_dir / "FBL_template_chunks.parquet"



fbl_chunks_df.to_parquet(
    chunks_parquet_path,
    index=False
)

print("FBL artifacts saved successfully:")
print(extracted_path)
print(cleaned_path)
print(chunks_parquet_path)

FBL artifacts saved successfully:
C:\Users\terbo\Desktop\Project\data\processed\rag\extracted\FBL_template_reconstructed.txt
C:\Users\terbo\Desktop\Project\data\processed\rag\cleaned\FBL_template_logical_units.json
C:\Users\terbo\Desktop\Project\data\processed\rag\chunks\FBL_template_chunks.parquet


### 7.1 Reload and Verify Saved FBL Chunks

Reload the chunks that were just written to disk and re-run the shared schema validator against them. This confirms the parquet file on disk — not just the in-memory DataFrame — satisfies the canonical schema.

In [37]:
import pandas as pd

chunks_path = r"C:\Users\terbo\Desktop\Project\data\processed\rag\chunks\FBL_template_chunks.parquet"

chunks_df = pd.read_parquet(chunks_path)

validate_chunk_schema(chunks_df, "FBL_chunks (reloaded from disk)")

print(f"Total chunks: {len(chunks_df)}")
display(chunks_df)


--- Schema validation: FBL_chunks (reloaded from disk) ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] sub_chunk_requires_parent_chunk
FBL_chunks (reloaded from disk): all checks passed (31 chunks)

Total chunks: 31


,page,unit_type,logical_unit,chunk_id,parent_chunk,sub_chunk,title,text,char_count,filename,source,real_or_synthetic,document_type,department,sensitivity_level,version_date
0,1,form,fbl_form,fbl_form_chunk_1,<NA>,None,Negotiable FIATA Multimodal Transport Bill of ...,Consignor\nFBL\nNEGOTIABLE FIATA MULTIMODAL TR...,1475,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
1,2,definitions,definitions,definitions_chunk_1,<NA>,None,Definitions,Definitions\n– «Freight Forwarder» means the M...,964,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
2,2,clause,clause_1,clause_1_chunk_1,1.0,None,Applicability,1. Applicability\nNotwithstanding the heading ...,167,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
3,2,clause,clause_2,clause_2_chunk_1,2.0,None,Issuance of this FBL,2. Issuance of this FBL\n2.1. By issuance of t...,711,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
4,2,clause,clause_3,clause_3_chunk_1,3.0,None,Negotiability and title to the goods,3. Negotiability and title to the goods\n3.1. ...,804,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
5,2,clause,clause_4,clause_4_chunk_1,4.0,None,Dangerous Goods and Indemnity,4. Dangerous Goods and Indemnity\n4.1. The Mer...,1550,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
6,2,clause,clause_5,clause_5_chunk_1,5.0,None,Description of Goods and Merchant’s Packing an...,5. Description of Goods and Merchant’s Packing...,1570,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
7,2,clause,clause_6,clause_6_chunk_1,6.0,"[6.1, 6.2, 6.3, 6.4]",Freight Forwarder’s Liability,6. Freight Forwarder’s Liability\n\n6.1. The r...,1502,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
8,2,clause,clause_6,clause_6_chunk_2,6.0,[6.5],Freight Forwarder’s Liability,6. Freight Forwarder’s Liability\n\n6.5. When ...,919,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>
9,2,clause,clause_6,clause_6_chunk_3,6.0,[6.6],Freight Forwarder’s Liability,6. Freight Forwarder’s Liability\n\n6.6. Defen...,786,FBL_template.pdf,FIATA,real,bill_of_lading_template,Operations,1,<NA>


## Step 1 — Setup

In this step, we import the required libraries and define the input
path for the Suez Canal Transit Dues document.

We will process only Page 4, which contains the "Notes" section.

In [38]:
import re
import pandas as pd
from pathlib import Path

# Input PDF
pdf_path = Path("C:\\Users\\terbo\\Desktop\\Project\\Data\\raw\\unstructured\\Transit_Dues_Rates.pdf")

print("PDF exists:", pdf_path.exists())


PDF exists: True


## Step 2 — Extract the Notes section

The Suez PDF contains four pages, but only Page 4 is relevant for this source.

We intentionally exclude the vessel-rate tables and extract only the section
starting from "Notes:".

No semantic rewriting is performed at this stage.

In [39]:
import fitz  # PyMuPDF

doc = fitz.open(pdf_path)

print(f"Number of pages: {len(doc)}")

# Page 4 = index 3
page_4_text = doc[3].get_text("text")

print(page_4_text)

Number of pages: 4
9 
 
 
 
13- Other Vessels: 
Suez Canal Net Tonnage (SCNT)                                                                                                                                                         SDR/SCNT 
First 5000 Tons 
Next 5000 Tons 
Next 10000 Tons 
Next 20000 Tons 
Next 30000 Tons 
Next 50000 Tons 
The Rest 
Laden 
Ballast 
Laden 
Ballast 
Laden 
Ballast 
Laden 
Ballast 
Laden 
Ballast 
Laden 
Ballast 
Laden 
Ballast 
10.04 
8.53 
6.76 
5.75 
5.69 
4.84 
4.14 
3.52 
4.01 
3.42 
3.75 
3.20 
3.62 
3.07 
 
Notes: 
- 
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of 
cargo is carried the carrier shall be charged with the highest transit

In [40]:
from pathlib import Path

extracted_path = Path(
    r"C:\Users\terbo\Desktop\Project\Data\processed\rag\extracted\suez_notes_extracted.txt"
)

extracted_path.write_text(
    page_4_text,
    encoding="utf-8"
)

print("Extracted text saved to:")
print(extracted_path)

Extracted text saved to:
C:\Users\terbo\Desktop\Project\Data\processed\rag\extracted\suez_notes_extracted.txt


## Step 3 — Isolate the Notes section

We locate the "Notes:" heading and discard everything before it.

This guarantees that the rate tables from Page 4 are not accidentally included
in the Suez Notes chunks.

In [41]:
notes_start = page_4_text.find("Notes:")

if notes_start == -1:
    raise ValueError("Could not find 'Notes:' in Page 4.")

notes_text = page_4_text[notes_start:].strip()

print(notes_text)

Notes: 
- 
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of 
cargo is carried the carrier shall be charged with the highest transit dues rate among the carried cargo. 
- 
Ballast “Combined Carriers (OBO)” shall be charged with transit dues rates of ballast “Dry Bulk Vessels”. 
- 
Vessels (Navy or Commercial) refusing or abstaining from upon-transit inspection, the following shall be applied: 
a. Suez Canal transit dues will be calculated and collected on the basis of Suez Canal Gross Tonnage.  
b. “Additional charges” specified and levied at the time of transit shall be collected including (any additional dues related to the type of vessel, the 
highest additional dues rel

## Step 4 — Clean the Notes text

The cleaning here is intentionally conservative.

We only normalize:
- excessive whitespace
- line breaks caused by PDF extraction
- spacing around the text

We do NOT rewrite the regulatory content or change terminology.

In [42]:
def clean_pdf_text(text):
    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


clean_notes_text = clean_pdf_text(notes_text)

print(clean_notes_text)

Notes: 
- 
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of 
cargo is carried the carrier shall be charged with the highest transit dues rate among the carried cargo. 
- 
Ballast “Combined Carriers (OBO)” shall be charged with transit dues rates of ballast “Dry Bulk Vessels”. 
- 
Vessels (Navy or Commercial) refusing or abstaining from upon-transit inspection, the following shall be applied: 
a. Suez Canal transit dues will be calculated and collected on the basis of Suez Canal Gross Tonnage. 
b. “Additional charges” specified and levied at the time of transit shall be collected including (any additional dues related to the type of vessel, the 
highest additional dues rela

## Step 5 — Save the Cleaned Text

Persist the cleaned Notes text to disk before structural parsing begins, so this intermediate result can be inspected or reused independently of the chunking logic below.

In [43]:
from pathlib import Path

cleaned_path = Path(
    r"C:\Users\terbo\Desktop\Project\Data\processed\rag\cleaned\suez_notes_cleaned.txt"
)

cleaned_path.parent.mkdir(parents=True, exist_ok=True)

cleaned_path.write_text(
    clean_notes_text,
    encoding="utf-8"
)

print("Cleaned text saved to:")
print(cleaned_path)

Cleaned text saved to:
C:\Users\terbo\Desktop\Project\Data\processed\rag\cleaned\suez_notes_cleaned.txt


## Step 6 — Structure-Aware Logical Unit Detection

Instead of relying only on the extracted text, use the PDF's
text-block structure to preserve natural logical boundaries.

The pipeline should:

1. Extract text blocks with their layout information.
2. Detect bullet/number markers.
3. Keep wrapped lines belonging to the same item together.
4. Preserve nested items such as a./b. inside their parent rule.
5. Avoid splitting list continuations into independent logical units.

No source-specific phrases are hardcoded.

In [44]:
import fitz
import re

doc = fitz.open(pdf_path)
page = doc[3]

blocks = page.get_text("blocks")

print("Detected blocks:", len(blocks))

for i, block in enumerate(blocks):
    x0, y0, x1, y1, text, block_no, block_type = block[:7]

    print(f"\n--- Block {i} ---")
    print(f"y={y0:.2f} → {y1:.2f}")
    print(repr(text))

Detected blocks: 7

--- Block 0 ---
y=576.79 → 608.44
'9 \n \n'

--- Block 1 ---
y=21.91 → 88.64
' \n \n13- Other Vessels: \nSuez Canal Net Tonnage (SCNT)                                                                                                                                                         SDR/SCNT \n'

--- Block 2 ---
y=90.69 → 118.28
'First 5000 Tons \nNext 5000 Tons \nNext 10000 Tons \nNext 20000 Tons \nNext 30000 Tons \nNext 50000 Tons \nThe Rest \nLaden \nBallast \nLaden \nBallast \nLaden \nBallast \nLaden \nBallast \nLaden \nBallast \nLaden \nBallast \nLaden \nBallast \n'

--- Block 3 ---
y=126.57 → 139.88
'10.04 \n8.53 \n6.76 \n5.75 \n5.69 \n4.84 \n4.14 \n3.52 \n4.01 \n3.42 \n3.75 \n3.20 \n3.62 \n3.07 \n'

--- Block 4 ---
y=149.11 → 301.40
' \nNotes: \n- \nBallast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. \n- \nBallast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil T

## Step 7 — Extract Logical Units from PDF Layout

Use PDF text blocks as the structural source of truth.

A logical unit starts when a block contains a bullet marker (`-`)
and continues across subsequent lines/blocks until the next top-level
bullet marker.

Nested markers such as `a.` and `b.` remain part of their parent unit.

This approach is source-agnostic and preserves the original source text.

In [45]:
def extract_note_blocks(page):
    """
    Extract text blocks belonging to the Notes section.

    Blocks are ordered by their vertical position on the page.
    """

    blocks = page.get_text("blocks")

    note_blocks = []

    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block[:7]

        text = text.strip()

        if not text:
            continue

        # Start collecting once Notes: appears
        if "Notes:" in text:
            note_blocks.append(text[text.find("Notes:"):].strip())
            continue

        if note_blocks:
            note_blocks.append(text)

    return note_blocks


note_blocks = extract_note_blocks(page)

print("Detected note blocks:", len(note_blocks))

for i, block in enumerate(note_blocks, 1):
    print(f"\n--- Note Block {i} ---")
    print(block)

Detected note blocks: 3

--- Note Block 1 ---
Notes: 
- 
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”. 
- 
Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of 
cargo is carried the carrier shall be charged with the highest transit dues rate among the carried cargo. 
- 
Ballast “Combined Carriers (OBO)” shall be charged with transit dues rates of ballast “Dry Bulk Vessels”. 
- 
Vessels (Navy or Commercial) refusing or abstaining from upon-transit inspection, the following shall be applied: 
a. Suez Canal transit dues will be calculated and collected on the basis of Suez Canal Gross Tonnage.  
b. “Additional charges” specified and levied at the time of transit shall be collected including (any additional dues related to the ty

## Step 8 — Reconstruct Top-Level Logical Units

Reconstruct logical units from the extracted PDF blocks.

The parser detects top-level bullet markers while preserving
continuations across lines and blocks.

Nested items such as `a.` and `b.` are not treated as separate
logical units.

In [46]:
def reconstruct_logical_units(note_blocks):
    """
    Reconstruct top-level logical units from PDF blocks.

    A top-level '-' starts a new logical unit.
    Nested markers such as 'a.' and 'b.' remain inside
    the current logical unit.

    No source-specific content is used.
    """

    # Combine blocks while preserving their textual order
    full_text = "\n".join(note_blocks)

    # Normalize line endings
    full_text = full_text.replace("\r\n", "\n").replace("\r", "\n")

    # Normalize spaces but keep newlines temporarily
    full_text = re.sub(r"[ \t]+", " ", full_text)

    lines = [
        line.strip()
        for line in full_text.splitlines()
        if line.strip()
    ]

    units = []
    current = None

    for line in lines:

        # Ignore Notes heading
        if line.lower() == "notes:":
            continue

        # Top-level bullet
        is_top_level_bullet = re.match(r"^-\s*$", line) or re.match(
            r"^-\s+",
            line
        )

        if is_top_level_bullet:

            # Save previous unit
            if current is not None:
                units.append({
                    "text": " ".join(current).strip()
                })

            # Remove bullet marker
            content = re.sub(r"^-\s*", "", line).strip()

            current = []

            if content:
                current.append(content)

        else:
            # Continuation of current unit
            if current is not None:
                current.append(line)

    # Save final unit
    if current is not None:
        units.append({
            "text": " ".join(current).strip()
        })

    return units


logical_units = reconstruct_logical_units(note_blocks)

print("Detected logical units:", len(logical_units))

for i, unit in enumerate(logical_units, 1):
    print(f"\n--- Logical Unit {i} ---")
    print(unit["text"])

Detected logical units: 8

--- Logical Unit 1 ---
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”.

--- Logical Unit 2 ---
Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”.

--- Logical Unit 3 ---
Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of cargo is carried the carrier shall be charged with the highest transit dues rate among the carried cargo.

--- Logical Unit 4 ---
Ballast “Combined Carriers (OBO)” shall be charged with transit dues rates of ballast “Dry Bulk Vessels”.

--- Logical Unit 5 ---
Vessels (Navy or Commercial) refusing or abstaining from upon-transit inspection, the following shall be applied: a. Suez Canal transit dues will be calculated and collected on the basis of Suez Canal Gross Tonnage. b. “Additional charges” specified and levied at the time of transi

## Step 9 — Semantic Grouping of Logical Units

Group adjacent logical units when they are semantically related
and the resulting chunk remains within the configured size limit.

Logical units are never split internally.

The algorithm therefore preserves structural boundaries while allowing
related units to form larger retrieval chunks.

No source-specific content or rules are hardcoded.

In [47]:
# %pip install -q scikit-learn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def semantic_group_units(
    units,
    max_chars=1800,
    similarity_threshold=0.10
):
    """
    Group adjacent logical units.

    Rules:
    - Never split a logical unit.
    - Only adjacent units can be grouped.
    - Do not exceed max_chars.
    - Use semantic similarity to decide whether adjacent
      units belong together.
    """

    if not units:
        return []

    texts = [unit["text"] for unit in units]

    # TF-IDF is used only for measuring similarity between
    # already detected logical units.
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        lowercase=True
    )

    vectors = vectorizer.fit_transform(texts)

    grouped = []
    current_group = [units[0]]

    for i in range(1, len(units)):

        current_text = "\n\n".join(
            unit["text"] for unit in current_group
        )

        next_text = units[i]["text"]

        candidate_text = (
            current_text
            + "\n\n"
            + next_text
        )

        similarity = cosine_similarity(
            vectors[i - 1],
            vectors[i]
        )[0][0]

        exceeds_size = len(candidate_text) > max_chars
        weak_similarity = similarity < similarity_threshold

        if exceeds_size or weak_similarity:

            grouped.append(current_group)

            current_group = [units[i]]

        else:
            current_group.append(units[i])

    grouped.append(current_group)

    return grouped

In [48]:
semantic_groups = semantic_group_units(
    logical_units,
    max_chars=1800,
    similarity_threshold=0.10
)

print("Final semantic groups:", len(semantic_groups))

for i, group in enumerate(semantic_groups, 1):

    text = "\n\n".join(
        unit["text"] for unit in group
    )

    print(f"\n--- Semantic Group {i} ---")
    print("Units:", len(group))
    print("Characters:", len(text))
    print(text)

Final semantic groups: 4

--- Semantic Group 1 ---
Units: 4
Characters: 574
Ballast “Petroleum Product Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”.

Ballast “Oil/Chemical Tankers” shall be charged with transit dues rates of ballast “Crude Oil Tankers”.

Laden “Combined Carriers (OBO)” shall be charged with transit dues rates according to the type of the carried cargo. In case more than one type of cargo is carried the carrier shall be charged with the highest transit dues rate among the carried cargo.

Ballast “Combined Carriers (OBO)” shall be charged with transit dues rates of ballast “Dry Bulk Vessels”.

--- Semantic Group 2 ---
Units: 1
Characters: 479
Vessels (Navy or Commercial) refusing or abstaining from upon-transit inspection, the following shall be applied: a. Suez Canal transit dues will be calculated and collected on the basis of Suez Canal Gross Tonnage. b. “Additional charges” specified and levied at the time of transit shall be colle

## Step 10 — Build Final Chunks with Unified Metadata

Convert the semantic groups into final chunks using the common
project-wide schema.

The source text remains unchanged.

Source and RBAC metadata are stored for validation and retrieval
filtering, not as part of the generated answer — the same applies to every identifying field (including `filename` and `chunk_id`): none of them should be echoed verbatim into a user-facing answer unless the design explicitly calls for citing sources.

In [49]:
def build_chunk_dataframe(
    semantic_groups,
    page,
    filename,
    source,
    real_or_synthetic,
    document_type,
    department,
    sensitivity_level,
    version_date=None,
    chunk_descriptors=None
):
    """
    Build final chunks using the unified project schema.

    Source-specific values and optional descriptors are metadata.
    The chunk text itself remains source-faithful.
    """

    if (
        chunk_descriptors is not None
        and len(chunk_descriptors) != len(semantic_groups)
    ):
        raise ValueError(
            "Each semantic group needs one chunk descriptor."
        )

    rows = []

    for idx, group in enumerate(semantic_groups, start=1):

        descriptor = (
            chunk_descriptors[idx - 1]
            if chunk_descriptors is not None
            else {}
        )

        text = "\n\n".join(
            unit["text"]
            for unit in group
        ).strip()

        rows.append({
            "page": page,
            "unit_type": "note",
            "logical_unit": descriptor.get(
                "logical_unit", f"notes_group_{idx}"
            ),
            "chunk_id": descriptor.get(
                "chunk_id", f"suez_notes_chunk_{idx}"
            ),

            # Explicit nullable values. The Notes section has no
            # parent/sub-chunk hierarchy, so both stay empty.
            "parent_chunk": pd.NA,
            "sub_chunk": None,
            "title": descriptor.get("title", pd.NA),

            "text": text,
            "char_count": len(text),

            # Metadata
            "filename": filename,
            "source": source,
            "real_or_synthetic": real_or_synthetic,
            "document_type": document_type,
            "department": department,
            "sensitivity_level": sensitivity_level,
            "version_date": version_date
        })

    # Apply the same schema used for FBL. This only standardizes
    # field order and nullable types; it does not rewrite chunk data.
    return normalize_chunk_schema(pd.DataFrame(rows))


In [50]:
# Descriptive metadata improves standalone retrieval without changing text.
suez_chunk_descriptors = [
    {
        "logical_unit": "ballast_and_combined_carrier_rate_rules",
        "chunk_id": "suez_notes_ballast_combined_carrier_rates",
        "title": "Ballast and Combined Carrier Transit-Dues Rules",
    },
    {
        "logical_unit": "inspection_refusal_dues_and_additional_charges",
        "chunk_id": "suez_notes_inspection_refusal_dues",
        "title": "Transit Inspection Refusal, Dues, and Additional Charges",
    },
    {
        "logical_unit": "special_floating_units_definition",
        "chunk_id": "suez_notes_special_floating_units",
        "title": "Definition of Special Floating Units",
    },
    {
        "logical_unit": "transit_conditions_and_accepted_payment_currencies",
        "chunk_id": "suez_notes_transit_conditions_payment_currencies",
        "title": "Transit Conditions and Accepted Payment Currencies",
    },
]

suez_chunks_df = build_chunk_dataframe(
    semantic_groups=semantic_groups,
    page=4,
    filename="Transit_Dues_Rates.pdf",
    source="Suez Canal Authority",
    real_or_synthetic="real",
    document_type="transit_dues_notes",
    department="Operations",
    sensitivity_level=1,
    version_date=None,
    chunk_descriptors=suez_chunk_descriptors
)

suez_chunks_df

,page,unit_type,logical_unit,chunk_id,parent_chunk,sub_chunk,title,text,char_count,filename,source,real_or_synthetic,document_type,department,sensitivity_level,version_date
0,4,note,ballast_and_combined_carrier_rate_rules,suez_notes_ballast_combined_carrier_rates,<NA>,None,Ballast and Combined Carrier Transit-Dues Rules,Ballast “Petroleum Product Tankers” shall be c...,574,Transit_Dues_Rates.pdf,Suez Canal Authority,real,transit_dues_notes,Operations,1,<NA>
1,4,note,inspection_refusal_dues_and_additional_charges,suez_notes_inspection_refusal_dues,<NA>,None,"Transit Inspection Refusal, Dues, and Addition...",Vessels (Navy or Commercial) refusing or absta...,479,Transit_Dues_Rates.pdf,Suez Canal Authority,real,transit_dues_notes,Operations,1,<NA>
2,4,note,special_floating_units_definition,suez_notes_special_floating_units,<NA>,None,Definition of Special Floating Units,“Special Floating Units” includes the followin...,323,Transit_Dues_Rates.pdf,Suez Canal Authority,real,transit_dues_notes,Operations,1,<NA>
3,4,note,transit_conditions_and_accepted_payment_curren...,suez_notes_transit_conditions_payment_currencies,<NA>,None,Transit Conditions and Accepted Payment Curren...,The aforementioned transit dues rates shall be...,478,Transit_Dues_Rates.pdf,Suez Canal Authority,real,transit_dues_notes,Operations,1,<NA>


## Step 11 — Validate the Unified Chunk Schema

Run the same shared validator used for the FBL chunks — this is the Suez_chunk test — to confirm this dataset follows the identical schema used across the project.

Validation includes:

- required columns
- missing values
- chunk IDs
- character counts
- sensitivity levels
- source metadata

In [51]:
validate_chunk_schema(suez_chunks_df, "Suez_chunks")

print("Rows:", len(suez_chunks_df))
print("Columns:", len(suez_chunks_df.columns))


--- Schema validation: Suez_chunks ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] sub_chunk_requires_parent_chunk
Suez_chunks: all checks passed (4 chunks)

Rows: 4
Columns: 16


## Step 12 — Save Suez Chunks

Persist the final Suez Notes chunks so they can be combined with the FBL chunks (and any future resource) into the unified RAG chunks table.

In [52]:
chunks_path = Path(
    r"C:\Users\terbo\Desktop\Project\Data\processed\rag\chunks\suez_notes_chunks.parquet"
)

chunks_path.parent.mkdir(parents=True, exist_ok=True)

suez_chunks_df.to_parquet(
    chunks_path,
    index=False
)

print("Final chunks saved to:")
print(chunks_path)

print("Final chunks:", len(suez_chunks_df))

Final chunks saved to:
C:\Users\terbo\Desktop\Project\Data\processed\rag\chunks\suez_notes_chunks.parquet
Final chunks: 4


## Final Validation — Unified Chunk Schema & Cross-Source Tests

Reload both chunk sets from disk and run the shared schema validator against each one again — this re-confirms the FBL_chunk and Suez_chunk tests against what was actually persisted, not just the in-memory DataFrames. Then check that both datasets share identical column names, order, and dtypes, so they can be concatenated safely into the unified RAG chunks table.

In [53]:
from pathlib import Path
import pandas as pd


# Paths
fbl_chunks_path = Path(
    r"C:\Users\terbo\Desktop\Project\Data\processed\rag\chunks\FBL_template_chunks.parquet"
)

suez_chunks_path = Path(
    r"C:\Users\terbo\Desktop\Project\Data\processed\rag\chunks\suez_notes_chunks.parquet"
)


# Load chunks
fbl_df = pd.read_parquet(fbl_chunks_path)
suez_df = pd.read_parquet(suez_chunks_path)


print("FBL shape:", fbl_df.shape)
print("Suez shape:", suez_df.shape)

FBL shape: (31, 16)
Suez shape: (4, 16)


In [54]:
fbl_columns = list(fbl_df.columns)
suez_columns = list(suez_df.columns)

print("FBL columns:")
print(fbl_columns)

print("\nSuez columns:")
print(suez_columns)

print("\nSame columns and order:", fbl_columns == suez_columns)

FBL columns:
['page', 'unit_type', 'logical_unit', 'chunk_id', 'parent_chunk', 'sub_chunk', 'title', 'text', 'char_count', 'filename', 'source', 'real_or_synthetic', 'document_type', 'department', 'sensitivity_level', 'version_date']

Suez columns:
['page', 'unit_type', 'logical_unit', 'chunk_id', 'parent_chunk', 'sub_chunk', 'title', 'text', 'char_count', 'filename', 'source', 'real_or_synthetic', 'document_type', 'department', 'sensitivity_level', 'version_date']

Same columns and order: True


In [55]:
schema_comparison = pd.DataFrame({
    "FBL_dtype": fbl_df.dtypes.astype(str),
    "Suez_dtype": suez_df.dtypes.astype(str)
})

schema_comparison["same_dtype"] = (
    schema_comparison["FBL_dtype"]
    == schema_comparison["Suez_dtype"]
)

schema_comparison

,FBL_dtype,Suez_dtype,same_dtype
page,Int64,Int64,True
unit_type,string,string,True
logical_unit,string,string,True
chunk_id,string,string,True
parent_chunk,string,string,True
sub_chunk,object,object,True
title,string,string,True
text,string,string,True
char_count,Int64,Int64,True
filename,string,string,True


In [56]:
# Re-validate the persisted (parquet) copies using the same shared
# validator that was used right after each dataset was built. This is
# the FBL_chunk / Suez_chunk test against what is actually on disk.
validate_chunk_schema(fbl_df, "FBL_chunks (reloaded)")
validate_chunk_schema(suez_df, "Suez_chunks (reloaded)")

# Both datasets must share the exact same schema before they can be
# concatenated into the unified RAG chunks table.
assert fbl_columns == suez_columns, (
    "Schema validation failed: column names or order differ."
)
assert fbl_df.dtypes.astype(str).equals(suez_df.dtypes.astype(str)), (
    "Schema validation failed: FBL and Suez dtypes differ."
)

# A non-PDF probe proves the schema stays stable for a resource type
# that has no pages, parent/sub-chunk relationships, or a title.
probe_text = "Example audio transcript segment."
resource_schema_probe = normalize_chunk_schema(
    pd.DataFrame([
        {
            "unit_type": "transcript_segment",
            "logical_unit": "speaker_turn_001",
            "chunk_id": "audio_call_speaker_turn_001",
            "text": probe_text,
            "char_count": len(probe_text),
            "filename": "operations_call.mp3",
            "source": "Operations",
            "real_or_synthetic": "real",
            "document_type": "audio_transcript",
            "department": "Operations",
            "sensitivity_level": 1,
        }
    ])
)

validate_chunk_schema(resource_schema_probe, "Non-PDF schema probe")
assert resource_schema_probe["page"].isna().all()
assert resource_schema_probe["parent_chunk"].isna().all()

print("===================================")
print("UNIFIED SCHEMA VALIDATION: PASSED")
print("===================================")
print("Columns:", len(fbl_columns))
print("FBL rows:", len(fbl_df))
print("Suez rows:", len(suez_df))


--- Schema validation: FBL_chunks (reloaded) ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] sub_chunk_requires_parent_chunk
FBL_chunks (reloaded): all checks passed (31 chunks)

--- Schema validation: Suez_chunks (reloaded) ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] sub_chunk_requires_parent_chunk
Suez_chunks (reloaded): all checks passed (4 chunks)

--- Schema validation: Non-PDF schema probe ---
  [PASS] columns_match_schema
  [PASS] dtypes_match_schema
  [PASS] required_fields_complete
  [PASS] chunk_id_unique
  [PASS] text_non_empty
  [PASS] char_count_matches_text_length
  [PASS] sensitivity_level_in_range
  [PASS] 

### Note — Source Attribution in Generated Answers

The schema deliberately links every chunk to its source in metadata (`source`, `filename`, `chunk_id`) for retrieval filtering and RBAC — that part is settled and should stay as-is.

Separately, some chunk **text** itself names the issuing body (e.g. the Suez Notes chunk that quotes "Suez Canal Authority's rules and regulations"). That text must stay verbatim — rewriting or scrubbing a regulatory source's own wording would misrepresent it and is a compliance risk, not a cleanup task.

If a generic, non-attributing answer is a product requirement, it belongs in the retrieval/generation layer, not in this chunking pipeline:

- Do not forward raw `chunk_id` or `filename` values into the LLM's context — they are retrieval/RBAC metadata, not answer content.
- Add an explicit instruction in the generation system prompt telling the model to describe the regulation generically (e.g. "canal transit regulations") unless the user asks which authority issued it.
- If specific answers must never name the source even when asked, that is a prompt/business-logic decision to make deliberately, not a side effect of chunk cleaning.